## 1. Import Libraries and Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

## 2. Configure Plotting Style

In [ ]:
# Configure matplotlib parameters for better visualization
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 13

## 3. Data Discovery and Organization

In [ ]:
# Locate the data directory containing experiment results
data_dir = Path(__file__).parent / 'drem_experiments'

# Initialize dictionary to store file paths organized by DREM type
file_types = {'adapt': {}, 'standart': {}}

# Recursively search for CSV files and categorize them
for csv_file in data_dir.rglob('*.csv'):
    dataset_name = csv_file.parent.name
    if '_adapt.csv' in csv_file.name:
        file_types['adapt'][dataset_name] = csv_file
    elif '_standart.csv' in csv_file.name:
        file_types['standart'][dataset_name] = csv_file

## 4. Default Parameters for Data Slicing

In [ ]:
# Default parameters used as fixed values when analyzing the impact of other parameters
# When varying one parameter, others are fixed to these values
DEFAULT_FIXED_PARAMS = {
    'n_neurons': 10,
    'alpha': 0.0005,
    'epochs': 20,
    'n_samples': 1000,
    'n_features': 5,
    'noise': 1.0
}

## 5. Data Sanitization Function

In [ ]:
def sanitize_drem_data(df, threshold_multiplier=5):
    # Calculate baseline threshold as max of ADAM and SGD for each row
    max_baseline = df[['mse_adam', 'mse_sgd']].max(axis=1)
    threshold = max_baseline * threshold_multiplier
    
    # Identify outliers: DREM values exceeding the threshold
    outlier_mask = df['mse_drem'] > threshold
    num_outliers = outlier_mask.sum()
    
    # Replace outliers with ADAM MSE (reliable baseline)
    if num_outliers > 0:
        df.loc[outlier_mask, 'mse_drem'] = df.loc[outlier_mask, 'mse_adam']
    
    return df, num_outliers

## 6. Data Slicing Function

In [ ]:
def get_sliced_dataframe(df, vary_column, fixed_params):
    sliced_df = df.copy()
    
    # Iterate through fixed parameters and filter dataframe
    for col, target_val in fixed_params.items():
        # Skip the column that should vary
        if col == vary_column or col not in sliced_df.columns:
            continue
        
        # Skip if only one unique value exists
        unique_vals = sliced_df[col].unique()
        if len(unique_vals) <= 1:
            continue
        
        # Use exact value if available, otherwise find closest
        if target_val in unique_vals:
            sliced_df = sliced_df[sliced_df[col] == target_val]
        else:
            # Find nearest available value
            closest_val = unique_vals[np.argmin(np.abs(unique_vals - target_val))]
            sliced_df = sliced_df[sliced_df[col] == closest_val]
    
    return sliced_df

## 7. Graph Generation Function

In [ ]:
def create_graphs(drem_type, datasets_dict, output_suffix):
    # Load and preprocess all datasets
    datasets = {}
    for dataset_name, csv_file in datasets_dict.items():
        # Read CSV and standardize column names
        df = pd.read_csv(csv_file)
        df.columns = df.columns.str.strip().str.lower()
        # Remove outliers from DREM data
        df, _ = sanitize_drem_data(df)
        datasets[dataset_name] = df

    # Set up visualization style and color scheme
    plt.style.use('seaborn-v0_8-darkgrid')
    # Define colors for each optimizer for consistency across plots
    colors = {'DREM': '#1f77b4', 'ADAM': '#ff7f0e', 'SGD': '#2ca02c'}
    # Define markers for each optimizer for better visual distinction
    markers = {'DREM': 'o', 'ADAM': 's', 'SGD': '^'}

    # ========== Graph 1: MSE vs Epochs (all datasets) ==========
    fig, axes = plt.subplots(1, len(datasets), figsize=(20, 6))
    if len(datasets) == 1:
        axes = [axes]
        
    for idx, (dataset_name, df) in enumerate(datasets.items()):
        ax = axes[idx]
        # Filter data: keep epochs as varying parameter
        df_slice = get_sliced_dataframe(df, 'epochs', DEFAULT_FIXED_PARAMS)
        # Group by epochs and calculate mean MSE for each optimizer
        epochs_data = df_slice.groupby('epochs')[['mse_drem', 'mse_adam', 'mse_sgd']].mean().sort_index()
        
        # Plot performance curves
        ax.plot(epochs_data.index, epochs_data['mse_adam'], marker=markers['ADAM'], linewidth=3, markersize=10, label='ADAM', color=colors['ADAM'])
        ax.plot(epochs_data.index, epochs_data['mse_sgd'], marker=markers['SGD'], linewidth=3, markersize=10, label='SGD', color=colors['SGD'])
        ax.plot(epochs_data.index, epochs_data['mse_drem'], marker=markers['DREM'], linewidth=3, markersize=10, label=f'DREM ({drem_type})', color=colors['DREM'])
        
        # Configure axes labels and title
        ax.set_xlabel('Number of Epochs', fontsize=15, fontweight='bold')
        ax.set_ylabel('MSE', fontsize=15, fontweight='bold')
        ax.set_title(f'{dataset_name}', fontsize=16, fontweight='bold', pad=15)
        ax.legend(fontsize=13, loc='best')

    plt.tight_layout()
    plt.savefig(str(data_dir.parent / f'1_mse_vs_epochs{output_suffix}.png'), dpi=300, bbox_inches='tight')
    plt.close()


    if 'experiment_mr' in datasets:
        df_mr = datasets['experiment_mr']

        # ========== Graph 2: MSE vs Noise Level ==========
        fig, ax = plt.subplots(figsize=(13, 8))
        # Filter data: keep noise as varying parameter
        df_slice = get_sliced_dataframe(df_mr, 'noise', DEFAULT_FIXED_PARAMS)
        # Group by noise level and calculate mean MSE
        noise_data = df_slice.groupby('noise')[['mse_drem', 'mse_adam', 'mse_sgd']].mean().sort_index()
        
        # Plot noise impact analysis
        ax.plot(noise_data.index, noise_data['mse_adam'], marker=markers['ADAM'], linewidth=3, markersize=11, label='ADAM', color=colors['ADAM'])
        ax.plot(noise_data.index, noise_data['mse_sgd'], marker=markers['SGD'], linewidth=3, markersize=11, label='SGD', color=colors['SGD'])
        ax.plot(noise_data.index, noise_data['mse_drem'], marker=markers['DREM'], linewidth=3, markersize=11, label=f'DREM ({drem_type})', color=colors['DREM'])
        ax.set_xlabel('Noise Level', fontsize=16, fontweight='bold')
        ax.set_ylabel('MSE', fontsize=16, fontweight='bold')
        ax.set_title(f'Dependence of MSE on Noise Level, DREM={drem_type}', fontsize=18, fontweight='bold', pad=20)
        ax.legend(fontsize=14, loc='best')
        plt.savefig(str(data_dir.parent / f'2_mse_vs_noise{output_suffix}.png'), dpi=300, bbox_inches='tight')
        plt.close()

        # ========== Graph 3: MSE vs Sample Size ==========
        fig, ax = plt.subplots(figsize=(13, 8))
        # Filter data: keep sample size as varying parameter
        df_slice = get_sliced_dataframe(df_mr, 'n_samples', DEFAULT_FIXED_PARAMS)
        # Group by sample size and calculate mean MSE
        sample_data = df_slice.groupby('n_samples')[['mse_drem', 'mse_adam', 'mse_sgd']].mean().sort_index()
        
        # Plot sample size impact (logarithmic scale)
        ax.plot(sample_data.index, sample_data['mse_adam'], marker=markers['ADAM'], linewidth=3, markersize=11, label='ADAM', color=colors['ADAM'])
        ax.plot(sample_data.index, sample_data['mse_sgd'], marker=markers['SGD'], linewidth=3, markersize=11, label='SGD', color=colors['SGD'])
        ax.plot(sample_data.index, sample_data['mse_drem'], marker=markers['DREM'], linewidth=3, markersize=11, label=f'DREM ({drem_type})', color=colors['DREM'])
        ax.set_xlabel('Sample Size', fontsize=16, fontweight='bold')
        ax.set_ylabel('MSE', fontsize=16, fontweight='bold')
        ax.set_xscale('log')
        ax.set_title(f'Dependence of MSE on Sample Size, DREM={drem_type}', fontsize=18, fontweight='bold', pad=20)
        ax.legend(fontsize=14, loc='best')
        plt.savefig(str(data_dir.parent / f'3_mse_vs_sample_size{output_suffix}.png'), dpi=300, bbox_inches='tight')
        plt.close()

        # ========== Graph 4: MSE vs Number of Features ==========
        fig, ax = plt.subplots(figsize=(13, 8))
        # Filter data: keep feature count as varying parameter
        df_slice = get_sliced_dataframe(df_mr, 'n_features', DEFAULT_FIXED_PARAMS)
        # Group by number of features and calculate mean MSE
        features_data = df_slice.groupby('n_features')[['mse_drem', 'mse_adam', 'mse_sgd']].mean().sort_index()
        
        # Plot feature count impact
        ax.plot(features_data.index, features_data['mse_adam'], marker=markers['ADAM'], linewidth=3, markersize=11, label='ADAM', color=colors['ADAM'])
        ax.plot(features_data.index, features_data['mse_sgd'], marker=markers['SGD'], linewidth=3, markersize=11, label='SGD', color=colors['SGD'])
        ax.plot(features_data.index, features_data['mse_drem'], marker=markers['DREM'], linewidth=3, markersize=11, label=f'DREM ({drem_type})', color=colors['DREM'])
        ax.set_xlabel('Number of Features', fontsize=16, fontweight='bold')
        ax.set_ylabel('MSE', fontsize=16, fontweight='bold')
        ax.set_title(f'Dependence of MSE on Number of Features, DREM={drem_type}', fontsize=18, fontweight='bold', pad=20)
        ax.legend(fontsize=14, loc='best')
        plt.savefig(str(data_dir.parent / f'4_mse_vs_features{output_suffix}.png'), dpi=300, bbox_inches='tight')
        plt.close()

        # ========== Graph 5: Training Time vs Sample Size ==========
        fig, ax = plt.subplots(figsize=(13, 8))
        # Filter data: keep sample size as varying parameter
        df_slice = get_sliced_dataframe(df_mr, 'n_samples', DEFAULT_FIXED_PARAMS)
        # Group by sample size and calculate mean training time
        time_data = df_slice.groupby('n_samples')[['time_drem', 'time_adam', 'time_sgd']].mean().sort_index()
        
        # Plot computational efficiency analysis
        ax.plot(time_data.index, time_data['time_adam'], marker=markers['ADAM'], linewidth=3, markersize=11, label='ADAM', color=colors['ADAM'])
        ax.plot(time_data.index, time_data['time_sgd'], marker=markers['SGD'], linewidth=3, markersize=11, label='SGD', color=colors['SGD'])
        ax.plot(time_data.index, time_data['time_drem'], marker=markers['DREM'], linewidth=3, markersize=11, label=f'DREM ({drem_type})', color=colors['DREM'])
        ax.set_xlabel('Sample Size', fontsize=16, fontweight='bold')
        ax.set_ylabel('Training Time (seconds)', fontsize=16, fontweight='bold')
        ax.set_xscale('log')
        ax.set_title(f"Training Time vs Sample Size, DREM={drem_type})", fontsize=18, fontweight='bold', pad=20)
        ax.legend(fontsize=14, loc='best')
        plt.savefig(str(data_dir.parent / f'5_training_time_vs_sample_size{output_suffix}.png'), dpi=300, bbox_inches='tight')
        plt.close()

        # ========== Graph 6: Training Time vs Number of Features ==========
        fig, ax = plt.subplots(figsize=(13, 8))
        # Filter data: keep feature count as varying parameter
        df_slice = get_sliced_dataframe(df_mr, 'n_features', DEFAULT_FIXED_PARAMS)
        # Group by number of features and calculate mean training time
        time_data = df_slice.groupby('n_features')[['time_drem', 'time_adam', 'time_sgd']].mean().sort_index()
        
        # Plot feature count vs computational cost
        ax.plot(time_data.index, time_data['time_adam'], marker=markers['ADAM'], linewidth=3, markersize=11, label='ADAM', color=colors['ADAM'])
        ax.plot(time_data.index, time_data['time_sgd'], marker=markers['SGD'], linewidth=3, markersize=11, label='SGD', color=colors['SGD'])
        ax.plot(time_data.index, time_data['time_drem'], marker=markers['DREM'], linewidth=3, markersize=11, label=f'DREM ({drem_type})', color=colors['DREM'])
        ax.set_xlabel('Number of Features', fontsize=16, fontweight='bold')
        ax.set_ylabel('Training Time (seconds)', fontsize=16, fontweight='bold')
        ax.set_title(f"Training Time vs Number of Features, DREM={drem_type})", fontsize=18, fontweight='bold', pad=20)
        ax.legend(fontsize=14, loc='best')
        plt.savefig(str(data_dir.parent / f'6_training_time_vs_features{output_suffix}.png'), dpi=300, bbox_inches='tight')
        plt.close()

## 8. Execute Analysis

In [ ]:
# Generate plots for adaptive DREM implementation
if file_types['adapt']:
    print("Generating plots for adaptive DREM...")
    create_graphs('adapt', file_types['adapt'], '_adapt')
    print("Adaptive DREM plots completed.")

# Generate plots for standard DREM implementation
if file_types['standart']:
    print("Generating plots for standard DREM...")
    create_graphs('standart', file_types['standart'], '_standart')
    print("Standard DREM plots completed.")

print("Analysis complete! All visualization files have been saved.")